# Notebook 01 — Ingestion & Profiling

**Goal:** load the four raw CityTransit exports with explicit data types, profile each one, and record every data-quality problem that Notebook 02 has to fix.

**Input:** `data/raw/*.csv` **Output:** `data/interim/*_raw.parquet` (the same data, unchanged)

In [43]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

# Works whether the notebook is started from the project root or from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## 1. Explicit data types

Every identifier is read as a string so that values such as `R001` or `S0042` can never be turned into numbers, and the mixed boolean columns (`transfer_flag`, `has_shelter`) are also read as text so that values like `1`, `Y` and `TRUE` survive untouched until we normalise them on purpose. `passenger_count` uses the nullable `Int64` type because it contains missing values.

In [23]:
DTYPES = {
    "validations": {
        "validation_id": "string",
        "validation_ts": "string",
        "route_id": "string",
        "vehicle_id": "string",
        "stop_id": "string",
        "card_id": "string",
        "fare_type": "string",
        "fare_amount": "float64",
        "passenger_count": "Int64",
        "transfer_flag": "string",
    },
    "routes": {
        "route_id": "string",
        "route_name": "string",
        "mode": "string",
        "district": "string",
        "route_length_km": "float64",
        "scheduled_trips_per_day": "Int64",
    },
    "vehicles": {
        "vehicle_id": "string",
        "vehicle_type": "string",
        "capacity": "Int64",
        "year_manufactured": "Int64",
        "depot": "string",
    },
    "stops": {
        "stop_id": "string",
        "stop_name": "string",
        "district": "string",
        "zone": "string",
        "has_shelter": "string",
    },
}

## 2. Load the files

`load_raw` stops with a clear message if a file is missing (for example when the repository was cloned without the `data/raw` folder) instead of showing a long traceback.

In [24]:
class MissingInputError(Exception):
    """Raised when a required raw file is not found."""


def load_csv(path: Path, dtypes: dict) -> pd.DataFrame:
    if not path.exists():
        raise MissingInputError(
            f"Required input '{path.relative_to(PROJECT_ROOT)}' was not found. "
            "Place the four raw CSV exports in data/raw/ and run this notebook again."
        )
    return pd.read_csv(path, dtype=dtypes)


raw = {}
try:
    for name, dtypes in DTYPES.items():
        raw[name] = load_csv(RAW_DIR / f"{name}.csv", dtypes)
        print(f"Loaded {name:<12} {raw[name].shape[0]:>7,} rows x {raw[name].shape[1]} columns")
except MissingInputError as err:
    print(f"STOPPED: {err}")
    raise SystemExit(1)

Loaded validations   81,600 rows x 10 columns
Loaded routes            60 rows x 6 columns
Loaded vehicles         180 rows x 5 columns
Loaded stops            400 rows x 5 columns


The four row counts match the brief: about 81,600 validations (80,000 plus the injected duplicates), 60 routes, 180 vehicles and 400 stops.

The missing-file check, shown with a file name that does not exist:

In [25]:
try:
    load_csv(RAW_DIR / "example_missing_file.csv", {})
except MissingInputError as err:
    print(f"Handled cleanly -> {err}")

Handled cleanly -> Required input 'data\raw\example_missing_file.csv' was not found. Place the four raw CSV exports in data/raw/ and run this notebook again.


## 3. Profile of each file

`profile` returns one row per column with its dtype, null count and null percentage, and prints the overall shape and memory usage.

In [26]:
def profile(df: pd.DataFrame, name: str) -> pd.DataFrame:
    memory_kb = df.memory_usage(deep=True).sum() / 1024
    print(f"=== {name}: {df.shape[0]:,} rows, {df.shape[1]} columns, {memory_kb:,.1f} KB in memory ===")
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_count": df.isna().sum(),
        "null_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })

In [27]:
profile(raw["validations"], "validations")

=== validations: 81,600 rows, 10 columns, 11,385.1 KB in memory ===


,dtype,null_count,null_pct,n_unique
validation_id,string,0,0.00,80000
validation_ts,string,0,0.00,78622
route_id,string,0,0.00,121
vehicle_id,string,0,0.00,181
stop_id,string,0,0.00,401
card_id,string,9777,11.98,23501
fare_type,string,0,0.00,17
fare_amount,float64,1183,1.45,308
passenger_count,Int64,833,1.02,6
transfer_flag,string,0,0.00,6


In [28]:
raw["validations"].head()

,validation_id,validation_ts,route_id,vehicle_id,stop_id,card_id,fare_type,fare_amount,passenger_count,transfer_flag
0,VAL-0068918,2024-05-13 18:54:29,R001,V0044,S0234,CARD-001739,Full Fare,2.53,1,N
1,VAL-0044034,06/06/2024 08:48,R058,V0086,S0131,CARD-000729,ADULT,2.48,1,Y
2,VAL-0017450,06/03/2024 13:15,R043,V0177,S0251,CARD-020205,Adult,2.85,1,FALSE
3,VAL-0004736,2024-04-21 17:30:00,R026,V0108,S0330,CARD-013092,student,1.64,3,Y
4,VAL-0053735,2024-06-14 18:35:27,R003,V0057,S0389,CARD-003251,Student,1.34,1,0


**Validations — what the profile shows**

- `card_id` is empty for about 12% of rows. These are single-ride paper tickets, not errors.
- `fare_amount` (~1.5%) and `passenger_count` (~1%) have missing values.
- `validation_id` has fewer unique values than rows, so duplicate validations exist.
- `validation_ts` is stored as text and the first rows already show two different formats (`2024-05-13 18:54:29` and `06/06/2024 08:48`).

In [29]:
profile(raw["routes"], "routes")

=== routes: 60 rows, 6 columns, 5.0 KB in memory ===


,dtype,null_count,null_pct,n_unique
route_id,string,0,0.0,60
route_name,string,0,0.0,60
mode,string,0,0.0,10
district,string,3,5.0,10
route_length_km,float64,3,5.0,52
scheduled_trips_per_day,Int64,0,0.0,46


In [30]:
raw["routes"].head()

,route_id,route_name,mode,district,route_length_km,scheduled_trips_per_day
0,R001,Angel Harbor Line,tram,Harbourside,19.8,133
1,R002,Jeffrey Street Line,BUS,<NA>,12.6,80
2,R003,Erica Plains Line,METRO,Southbank,4.3,203
3,R004,Brittany Bypass Line,Tram,Northgate,17.6,131
4,R005,Clayton Fort Line,bus,<NA>,5.0,159


**Routes:** `district` and `route_length_km` each have 3 missing values (5%). `mode` uses mixed casing (`tram`, `BUS`, `METRO`), which will split one mode into several groups unless it is standardised.

In [31]:
profile(raw["vehicles"], "vehicles")

=== vehicles: 180 rows, 5 columns, 12.4 KB in memory ===


,dtype,null_count,null_pct,n_unique
vehicle_id,string,0,0.00,180
vehicle_type,string,0,0.00,4
capacity,Int64,12,6.67,87
year_manufactured,Int64,0,0.00,42
depot,string,0,0.00,4


In [32]:
raw["vehicles"].head()

,vehicle_id,vehicle_type,capacity,year_manufactured,depot
0,V0001,Articulated Bus,125,2021,Central Depot
1,V0002,Standard Bus,<NA>,1998,East Depot
2,V0003,Articulated Bus,118,2022,Central Depot
3,V0004,Tram,170,2008,Central Depot
4,V0005,Standard Bus,73,2013,Riverside Depot


**Vehicles:** 12 vehicles (6.7%) have no `capacity`, so a load factor cannot be calculated for them until the value is imputed. `year_manufactured` has no nulls but contains impossible years (checked below).

In [33]:
profile(raw["stops"], "stops")

=== stops: 400 rows, 5 columns, 31.1 KB in memory ===


,dtype,null_count,null_pct,n_unique
stop_id,string,0,0.0,400
stop_name,string,0,0.0,400
district,string,0,0.0,10
zone,string,0,0.0,3
has_shelter,string,0,0.0,4


In [34]:
raw["stops"].head()

,stop_id,stop_name,district,zone,has_shelter
0,S0001,matthew radial stop,Riverside,Zone 2,N
1,S0002,Brianna Avenue Stop,Centro,Zone 3,1
2,S0003,jean circle stop,Northgate,Zone 2,0
3,S0004,Carlos Bypass Stop,Old Town,Zone 1,Y
4,S0005,JONES FREEWAY STOP,Riverside,Zone 2,N


**Stops:** no missing values, but `stop_name` mixes lower case, upper case and title case, and `has_shelter` uses four different codes.

## 4. Categorical columns

In [35]:
def show_value_counts(df: pd.DataFrame, columns: list[str]) -> None:
    for col in columns:
        print(f"--- {col} ---")
        print(df[col].value_counts(dropna=False).to_string(), "\n")

In [36]:
show_value_counts(raw["validations"], ["fare_type", "transfer_flag"])

--- fare_type ---
fare_type
Full Fare         11405
ADULT             11317
Adult             11282
adult             11280
student            4077
STUDENT            4062
Student            3961
Senior             3251
Senior Citizen     3237
senior             3179
Kid                2202
child              2124
Child              2109
Pass Holder        2075
PassHolder         2032
Monthly Pass       2012
pass holder        1995 

--- transfer_flag ---
transfer_flag
N        36888
Y        24596
0         8075
FALSE     5593
1         4053
TRUE      2395 



**Problems in validations:**

- `fare_type` has 17 spellings for 5 real categories: casing variants (`Adult`/`adult`/`ADULT`) and synonyms (`Full Fare` = Adult, `Senior Citizen` = Senior, `Kid` = Child, `PassHolder`/`Monthly Pass` = Pass Holder).
- `transfer_flag` mixes three boolean encodings: `Y`/`N`, `1`/`0` and `TRUE`/`FALSE`.

In [37]:
v = raw["validations"]
ts = v["validation_ts"]
ts_formats = pd.Series({
    "ISO  YYYY-MM-DD HH:MM:SS": ts.str.match(r"^\d{4}-\d{2}-\d{2} ").sum(),
    "EU   DD/MM/YYYY HH:MM": ts.str.match(r"^\d{2}/\d{2}/\d{4} ").sum(),
    "Long Month D, YYYY H:MM AM": ts.str.match(r"^[A-Za-z]+ \d{1,2}, \d{4}").sum(),
})
print(ts_formats.to_string())
print("Unmatched:", len(ts) - ts_formats.sum())

eu = ts[ts.str.match(r"^\d{2}/\d{2}/\d{4} ")]
print("\nEU format — largest first part:", eu.str[:2].astype(int).max(),
      "| largest second part:", eu.str[3:5].astype(int).max())

ISO  YYYY-MM-DD HH:MM:SS      57316
EU   DD/MM/YYYY HH:MM         16169
Long Month D, YYYY H:MM AM     8115
Unmatched: 0

EU format — largest first part: 31 | largest second part: 6


**Timestamp formats:** three formats are present and every row matches one of them. In the `DD/MM/YYYY` rows the first number goes up to 31 and the second only up to 6, which proves the order is **day/month**. This matters: a generic parser reads `06/03/2024` as 3 June instead of 6 March, so Notebook 02 must parse each format with an explicit pattern.

In [38]:
print("Route IDs with leading/trailing spaces:", (v["route_id"] != v["route_id"].str.strip()).sum())
print("Exact duplicate rows:", v.duplicated().sum())
print("Duplicate validation_id:", v["validation_id"].duplicated().sum())
print("Negative passenger_count:", (v["passenger_count"] < 0).sum())
print("fare_amount above 100:", (v["fare_amount"] > 100).sum(),
      "-> values:", sorted(v.loc[v["fare_amount"] > 100, "fare_amount"].unique()))

Route IDs with leading/trailing spaces: 12092


Exact duplicate rows: 802
Duplicate validation_id: 1600
Negative passenger_count: 362
fare_amount above 100: 25 -> values: [np.float64(1234.56), np.float64(2000.0), np.float64(5000.0), np.float64(9999.0)]


**Other validation issues:** about 15% of `route_id` values carry a trailing space (they would fail to join to the routes table), 1,600 rows repeat an existing `validation_id` (802 of them are exact copies), a few hundred rows have negative passenger counts (reversed validations), and 25 fares are impossible values between 1,234 and 9,999.

In [39]:
show_value_counts(raw["routes"], ["mode", "district"])

--- mode ---
mode
bus       11
tram       9
BUS        9
Tram       9
Bus        8
BUS        6
metro      3
TRAM       2
Metro      2
METRO      1 

--- district ---
district
Centro         11
Westend         8
Northgate       6
Old Town        6
Hillcrest       6
Parkview        5
Harbourside     4
Southbank       4
Eastfield       4
<NA>            3
Riverside       3 



**Problems in routes:** `mode` has 10 spellings for 3 modes, including `BUS ` with a trailing space that looks identical to `BUS` when printed. `district` has 3 nulls that will need an explicit `Unknown` label.

In [40]:
show_value_counts(raw["vehicles"], ["vehicle_type", "depot"])
year = raw["vehicles"]["year_manufactured"]
print("year_manufactured outside 1980–2026:")
print(year[~year.between(1980, 2026)].value_counts().to_string())

--- vehicle_type ---
vehicle_type
Standard Bus       81
Articulated Bus    40
Tram               38
Metro Set          21 

--- depot ---
depot
Central Depot      54
East Depot         46
North Depot        45
Riverside Depot    35 

year_manufactured outside 1980–2026:
year_manufactured
2035    11
1899     4


**Problems in vehicles:** `vehicle_type` and `depot` are clean. 15 vehicles have impossible manufacturing years (`1899` or `2035`), which would give negative or 127-year ages.

In [41]:
show_value_counts(raw["stops"], ["zone", "has_shelter", "district"])
names = raw["stops"]["stop_name"]
print("Stop names with extra spaces:", (names != names.str.strip()).sum())
print("Stop names not in title case:", (names.str.strip() != names.str.strip().str.title()).sum())

--- zone ---
zone
Zone 1    213
Zone 2    108
Zone 3     79 

--- has_shelter ---
has_shelter
0    115
Y    102
N     96
1     87 

--- district ---
district
Northgate      48
Harbourside    45
Hillcrest      44
Riverside      41
Old Town       41
Westend        40
Centro         39
Parkview       36
Eastfield      36
Southbank      30 

Stop names with extra spaces: 103
Stop names not in title case: 146


**Problems in stops:** `zone` and `district` are clean. `has_shelter` mixes `Y`/`N` with `1`/`0`. Many stop names have extra spaces or are not in title case, so the same stop would display inconsistently in Power BI.

## 5. Save the raw frames unchanged

Parquet is used instead of CSV so that the explicit data types chosen above (strings for IDs, nullable integers) are kept when Notebook 02 reads the data back.

In [42]:
for name, df in raw.items():
    out = INTERIM_DIR / f"{name}_raw.parquet"
    df.to_parquet(out, index=False)
    print(f"Saved {out.relative_to(PROJECT_ROOT)}  ({len(df):,} rows)")

Saved data\interim\validations_raw.parquet  (81,600 rows)
Saved data\interim\routes_raw.parquet  (60 rows)
Saved data\interim\vehicles_raw.parquet  (180 rows)
Saved data\interim\stops_raw.parquet  (400 rows)


**Next:** `02_clean.ipynb` reads these four files and applies one cleaning function per rule.